# Portfolio Credit Risk

# Portfolio Credit Risk Analysis

## Background

We consider a portfolio with exposures against 100 different obligors (enumerated from 1 to 100). We define default indicator variables $\mathbb{I}_n$ for $n = 1, \ldots, 100$ as follows:

$$\mathbb{I}_n := \begin{cases} 1 & \text{if obligor } n \text{ defaults}^1 \text{ (} n = 1, \ldots, 100 \text{)} \\ 0 & \text{otherwise} \end{cases}$$

Furthermore, we assume that the exposure (expressed in USD) against obligor $n$ is equal to $c_n$, where $\sum_{n=1}^{100} c_n = 1,000$ (i.e., the aggregate portfolio exposure is equal to USD 1,000). In case of a default of obligor $n$, we assume that the entire exposure against this obligor (i.e., $c_n$) is lost.

The portfolio loss distribution $\mathcal{L}$ is given by:

$$\mathcal{L} = \sum_{n=1}^{100} c_n \cdot \mathbb{I}_n$$

We will analyse the following risk measures for this distribution:

- **Value-at-Risk (VaR)** for a certain confidence level (CL) $\gamma$ is defined as:

$$\text{VaR}_\gamma(\mathcal{L}) := \inf\{x \in \mathbb{R}: F_\mathcal{L}(x) \geq \gamma\} = \inf\{x \in \mathbb{R}: P(\mathcal{L} > x) \leq 1 - \gamma\}$$

where $F_\mathcal{L}(x)$ is the cumulative distribution function of $\mathcal{L}$. As an example, for a confidence level of 99%, the probability of observing a loss greater than VaR$_{99\%}(\mathcal{L})$ is at most 1%.

- **Tail Value-at-Risk (TVaR)** for a certain confidence level $\gamma$ is defined as:

$$\text{TVaR}_\gamma(\mathcal{L}) := \mathbb{E}[\mathcal{L} \mid \mathcal{L} \geq \text{VaR}_\gamma(\mathcal{L})]$$

Hence, TVaR is the expected loss given that an event outside a given confidence level has occurred.

- **The (unconditional) expected loss** $\mathbb{E}[\mathcal{L}]$ and the **standard deviation of the loss distribution** $\sigma[\mathcal{L}] = \sqrt{\mathbb{V}[\mathcal{L}]}$ ("loss volatility"), where $\mathbb{V}[\mathcal{L}]$ is the variance of the loss distribution.

$^1$ Within a certain time interval, e.g. one year, which is fixed throughout this case study.

## General Instructions

This case study aims to calculate the above risk measures under different assumptions in Python. For VaR and TVaR we will focus on the confidence levels (CLs) of 97%, 99% and 99.99%. Please provide a short report summarising your results, which should include the following elements:

- **Risk measure values**: Summarise the risk measure values in a table like the one below:

| Risk measures in USD | Exercise A VaR | Exercise A TVaR | Exercise B VaR | Exercise B TVaR | Exercise C VaR | Exercise C TVaR |
|---|---|---|---|---|---|---|
| CL = 97% | | | | | | |
| CL = 99% | | | | | | |
| CL = 99.99% | | | | | | |
| Expected loss | | | | | | |
| Loss volatility | | | | | | |

## Case Study

**Remark:** As specified below, the exposures from the file `exposures_B_and_C.csv` are only to be used for parts B and C.

**Hint:** You may simulate random variables using readily available libraries.

In [2]:
import numpy as np 
import pandas as pd 

In [3]:
df = pd.read_csv(r"C:\Users\Startklar\Documents\Bocconi - MAFINRISK\Milan\Becoming a quant\GitHub\BIS Assessment\Case study - exposures_B_and_C.csv")

In [4]:
df.head()

,Exposure
0,27.102751
1,3.812822
2,4.388454
3,0.280467
4,7.612454


### (A) Simulation 1: Independent Identically Distributed Defaults

Assume that $(\mathbb{I}_n)_{n=1,\ldots,100}$ are independent and identically Bernoulli($p = 1\%$) distributed and that $c_n = 10$ for all obligors $n = 1, \ldots, 100$.

Under these assumptions, estimate the above risk measures by a simulation approach with an appropriate number of simulations. Justify your choice of the number of simulations.

In [5]:
np.random.seed(42)

num_obligators = 100 
exposure_per_obligator = 10 
default_probability = 0.01
num_simulations = 10**6

simulated_losses = np.zeros(num_simulations)

for sim in range(num_simulations): 
        defaults = np.random.binomial(n=1, p=default_probability, size= num_obligators)

        loss = np.sum(exposure_per_obligator * defaults)

        simulated_losses[sim] = loss

expected_loss = np.mean(simulated_losses)
loss_volatility = np.std(simulated_losses)

results_A = {
        'Expected Loss': expected_loss, 
        'Loss volatility': loss_volatility, 
        'Var 97%': np.quantile(simulated_losses, 0.97), 
        'TVaR 97%': np.mean(simulated_losses[simulated_losses>= np.quantile(simulated_losses, 0.97)]),
        'Var 99%': np.quantile(simulated_losses, 0.99), 
        'TVaR 99%': np.mean(simulated_losses[simulated_losses>= np.quantile(simulated_losses, 0.99)]),
        'Var 99.99%': np.quantile(simulated_losses, 0.9999), 
        'TVaR 99.99%': np.mean(simulated_losses[simulated_losses>= np.quantile(simulated_losses, 0.9999)])
}

results_dfA = pd.DataFrame(results_A, index=[0]).T
results_dfA.columns = ['Exercise A']
print(results_dfA)

                 Exercise A
Expected Loss     10.011720
Loss volatility    9.945253
Var 97%           30.000000
TVaR 97%          32.780972
Var 99%           40.000000
TVaR 99%          42.191042
Var 99.99%        60.000000
TVaR 99.99%       61.503006


### (B) Simulation 2: Heterogeneous Exposures with Independent Defaults

Use the heterogeneous exposures $(c_n)_{n=1,\ldots,100}$ as provided in the file `"exposures_B_and_C.csv"`. As in (A), assume that $(\mathbb{I}_n)_{n=1,\ldots,100}$ are independent and identically Bernoulli($p = 1\%$) distributed.

Under these assumptions, estimate the above risk measures via a simulation approach.

In [7]:
exposures = df["Exposure"].values
num_obligators = len(exposures)
default_probability = 0.01
num_simulations = 10**6

simulated_losses_losees = np.zeros(num_simulations)

for sim in range(num_simulations):
    defaults = np.random.binomial(n=1, p=default_probability, size=num_obligators)

    loss = np.sum(exposures*defaults)

    simulated_losses[sim] = loss

expected_loss = np.mean(simulated_losses)
loss_volatility = np.std(simulated_losses)

results_B = {
        'Expected Loss': expected_loss, 
        'Loss volatility': loss_volatility, 
        'Var 97%': np.quantile(simulated_losses, 0.97), 
        'TVaR 97%': np.mean(simulated_losses[simulated_losses>= np.quantile(simulated_losses, 0.97)]),
        'Var 99%': np.quantile(simulated_losses, 0.99), 
        'TVaR 99%': np.mean(simulated_losses[simulated_losses>= np.quantile(simulated_losses, 0.99)]),
        'Var 99.99%': np.quantile(simulated_losses, 0.9999), 
        'TVaR 99.99%': np.mean(simulated_losses[simulated_losses>= np.quantile(simulated_losses, 0.9999)])
}

results_dfB = pd.DataFrame(results_B, index=[0]).T
results_dfB.columns = ['Exercise B']
print(results_dfB)

                 Exercise B
Expected Loss     10.002858
Loss volatility   13.476307
Var 97%           44.867240
TVaR 97%          55.380116
Var 99%           55.980109
TVaR 99%          67.104518
Var 99.99%       101.773794
TVaR 99.99%      110.140183


### (C) Simulation 3: Correlated Defaults via Beta-Binomial Model

As in (B), use the heterogeneous exposures $(c_n)_{n=1,\ldots,100}$ as provided in the file `"exposures_B_and_C.csv"`. However, model defaults as follows:

- Model the default probability $P$ via a Beta distribution$^3$ with parameters $\alpha = 0.2$ and $\beta = 19.8$
- Conditional on $P$, model $(\mathbb{I}_n)_{n=1,\ldots,100}$ as independent and identically Bernoulli($P$) distributed

Under these assumptions, (i) mathematically derive the correlation of any two default indicators $\mathbb{I}_n, \mathbb{I}_m$ ($n \neq m$) and (ii) estimate the above risk measures via a simulation approach.

**Hint:** In derivation (i), you can use the fact that the first two moments of the Beta distribution are given by:

$$\mathbb{E}[P] = \frac{\alpha}{\alpha + \beta} \quad \text{and} \quad \mathbb{E}[P^2] = \frac{\alpha(\alpha+1)}{(\alpha+\beta+1)(\alpha+\beta)}$$

$^3$ Note that the Beta distribution takes values in $(0, 1)$.

$^4$ This also implies that, as in A and B, the expected default probability is equal to $\mathbb{E}[P] = \frac{\alpha}{\alpha + \beta} = 1\%$.

In [8]:
# Part 1

exposures = df["Exposure"].values
num_obligators = len(exposures)
num_simulations = 10**6

alpha = 0.2 
beta = 19.8

simulated_losses_losees = np.zeros(num_simulations)

for sim in range(num_simulations):
    P = np.random.beta(alpha, beta)

    defaults = np.random.binomial(n=1, p=P, size=num_obligators)

    loss = np.sum(exposures*defaults)

    simulated_losses[sim] = loss

expected_loss = np.mean(simulated_losses)
loss_volatility = np.std(simulated_losses)

results_C = {
        'Expected Loss': expected_loss, 
        'Loss volatility': loss_volatility, 
        'Var 97%': np.quantile(simulated_losses, 0.97), 
        'TVaR 97%': np.mean(simulated_losses[simulated_losses>= np.quantile(simulated_losses, 0.97)]),
        'Var 99%': np.quantile(simulated_losses, 0.99), 
        'TVaR 99%': np.mean(simulated_losses[simulated_losses>= np.quantile(simulated_losses, 0.99)]),
        'Var 99.99%': np.quantile(simulated_losses, 0.9999), 
        'TVaR 99.99%': np.mean(simulated_losses[simulated_losses>= np.quantile(simulated_losses, 0.9999)])
}

results_dfC = pd.DataFrame(results_C, index=[0]).T
results_dfC.columns = ['Exercise C']
print(results_dfC)

                 Exercise C
Expected Loss     10.032006
Loss volatility   25.431388
Var 97%           79.164732
TVaR 97%         120.960279
Var 99%          124.833458
TVaR 99%         167.305767
Var 99.99%       322.174439
TVaR 99.99%      361.321512


In [9]:
# Part 2 

E_P = alpha/ (alpha + beta)
E_P2 = (alpha * (alpha+1)) / ((alpha+beta+1) * (alpha+beta))

cov = E_P2 - E_P**2
var = E_P - E_P**2
corr = cov / var

print(f"Expected value: {E_P}")
print(f"Covariance: {cov}")
print(f"Variance: {var}")
print(f"Correlation: {corr}")

Expected value: 0.01
Covariance: 0.00047142857142857137
Variance: 0.0099
Correlation: 0.04761904761904761


## Recap Table 

In [11]:
recap = pd.concat([results_dfA, results_dfB, results_dfC], axis=1)
recap

,Exercise A,Exercise B,Exercise C
Expected Loss,10.011720,10.002858,10.032006
Loss volatility,9.945253,13.476307,25.431388
Var 97%,30.000000,44.867240,79.164732
TVaR 97%,32.780972,55.380116,120.960279
Var 99%,40.000000,55.980109,124.833458
TVaR 99%,42.191042,67.104518,167.305767
Var 99.99%,60.000000,101.773794,322.174439
TVaR 99.99%,61.503006,110.140183,361.321512
